Use the requests lib to get the HTML of website. Even the status code return 403, but I still get the HTML of the website correctly. I'm not sure about what is going on, But I guess it may be some kind of the firewall to protect the actual website.

In [1]:
import requests

url = "https://www.ccny.cuny.edu/registrar/fall"
headers = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/153.0.0.0 Safari/537.36"
    )
}

response = requests.get(url, headers=headers)

print(response.status_code)
print(response.headers.get("Server"))
print(response.headers.get("Cf-Mitigated"))

403
cloudflare
challenge


After I do some research, I realize it will be the issues with the cloudflare. It protect the sever from the request that not come from the website. The request get some kind of challenge, so I can't access to the sever.

Since the get request will be stop by the cloudflare challenge. therefore the only thing I can do for the BeautifulSoup is copy the source page of the website down to the local and get the data from it.

In [2]:
from bs4 import BeautifulSoup

data =[]
with open("calendar.html", "r", encoding="utf-8") as f:
    html = f.read()

soup = BeautifulSoup(html, "html.parser")
table = soup.find("table")
rows = table.find_all("tr")

for row in rows:
    cells = row.find_all("td")

    if len(cells) == 3:
        date = cells[0].get_text(strip=True)
        dow = cells[1].get_text(strip=True)
        text = cells[2].get_text(strip=True)

        data.append({
            "date": date,
            "dow": dow,
            "text": text
        })

print(data[:3])


[{'date': 'August 01', 'dow': 'Sunday', 'text': 'Application for degree for January and February 2022 begins'}, {'date': 'August 18', 'dow': 'Wednesday', 'text': 'Last day to apply for Study Abroad'}, {'date': 'August 24', 'dow': 'Tuesday', 'text': 'Last day of Registration;Last day to file ePermit for the Fall 2021;Last day to drop classes for 100% tuition refund;'}]


Here I use the  BeautifulSoup's HTML parser to get the calendar data from local HTML page. I use the table, tr and td label in order to get the table data, each row in the table data and each cell of the row. I use the data list to keep each data from the calendar for late use.

Since we have the data list of the calendar, therefore, we just use the pandas lib to manage the data.

The function use to convert the date string into the python date

In [5]:
def parse_date(date_string):
    if "-" in date_string:
        date_string = date_string.split("-")[0].strip()

    if "," in date_string:
        return pd.to_datetime(date_string).date()

    return pd.to_datetime(date_string + ", 2021").date()

In [7]:
import pandas as pd

df = pd.DataFrame(data)

df["date"] = df["date"].apply(parse_date)
df = df.set_index("date")
df

,dow,text
date,,
2021-08-01,Sunday,Application for degree for January and Februar...
2021-08-18,Wednesday,Last day to apply for Study Abroad
2021-08-24,Tuesday,Last day of Registration;Last day to file ePer...
2021-08-25,Wednesday,Start of Fall Term;Classes begin;Initial Regis...
2021-08-25,Wednesday - Tuesday,Change of program period; late fees apply
2021-08-26,Thursday,Last day for Independent Study
2021-08-28,Saturday,First day of Saturday Classes
2021-08-31,Tuesday,Last day to add a class to an existing enrollm...
2021-09-01,Wednesday,Verification of Enrollment rosters available t...


For the last part, I use the data frame method of the pandas lib convert the data list into the data frame. And I apply the function that convert the date string into python date on the dataframe. Lastly I use set index to make the date become the index of the data frame.